In [ ]:
import boto3
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import io, boto3

In [ ]:
s3 = boto3.client('s3')
bucket = 'cust-seg-ygp'#bucket name
key = 'online_retail_II.csv'#file inside the bucket

obj = s3.get_object(Bucket=bucket, Key=key)
df = pd.read_csv(obj['Body'], low_memory=False)
print("Loaded rows:", len(df))
df.head()

Loaded rows: 1067371


In [ ]:
print("Shape (rows, columns):", df.shape)

Shape (rows, columns): (1067371, 8)


In [ ]:
print("Total rows:", df.shape[0])

Total rows: 1067371


In [ ]:
print("Column names:", df.columns.tolist())

Column names: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [ ]:
print(df.dtypes)

Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object


In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 34335


In [ ]:
# Show duplicate rows
dup_rows = df[df.duplicated()]
print(dup_rows.head())

    Invoice StockCode                        Description  Quantity  \
371  489517     21912           VINTAGE SNAKES & LADDERS         1   
383  489517     22130   PARTY CONE CHRISTMAS DECORATION          6   
384  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
385  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
386  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   

             InvoiceDate  Price  Customer ID         Country  
371  2009-12-01 11:34:00   3.75      16329.0  United Kingdom  
383  2009-12-01 11:34:00   0.85      16329.0  United Kingdom  
384  2009-12-01 11:34:00   0.65      16329.0  United Kingdom  
385  2009-12-01 11:34:00   3.75      16329.0  United Kingdom  
386  2009-12-01 11:34:00   3.75      16329.0  United Kingdom  


In [ ]:
df = df.drop_duplicates()

In [ ]:
print("After removing duplicates:", df.shape)

After removing duplicates: (1033036, 8)


In [ ]:
print(df.isnull().sum())

Invoice             0
StockCode           0
Description      4275
Quantity            0
InvoiceDate         0
Price               0
Customer ID    235151
Country             0
dtype: int64


In [ ]:
# View missing Customer ID rows BEFORE deleting
print(df['Customer ID'].isna().sum())
print(df[df['Customer ID'].isna()].head(5))

235151
    Invoice StockCode                Description  Quantity  \
263  489464     21733               85123a mixed       -96   
283  489463     71477                      short      -240   
284  489467    85123A                21733 mixed      -192   
470  489521     21646                        NaN       -50   
577  489525    85226C  BLUE PULL BACK RACING CAR         1   

             InvoiceDate  Price  Customer ID         Country  
263  2009-12-01 10:52:00   0.00          NaN  United Kingdom  
283  2009-12-01 10:52:00   0.00          NaN  United Kingdom  
284  2009-12-01 10:53:00   0.00          NaN  United Kingdom  
470  2009-12-01 11:44:00   0.00          NaN  United Kingdom  
577  2009-12-01 11:49:00   0.55          NaN  United Kingdom  


In [ ]:
# Drop rows with missing Customer ID
df = df.dropna(subset=["Customer ID"])
print("After dropping missing Customer ID rows:", df.shape)

After dropping missing Customer ID rows: (797885, 8)


In [ ]:
# Convert Customer ID from float to int
df["Customer ID"] = df["Customer ID"].astype(int)

In [ ]:
# View negative quantities BEFORE dropping
df[df['Quantity'] < 0].head(4)

In [ ]:
# Remove negative quantities (returns)
df = df[df['Quantity'] > 0]

In [ ]:
# View negative quantities AFTER dropping
df[df['Quantity'] < 0].head(4)

In [ ]:
# Remove zero/negative prices
print((df['Price'] <= 0).sum())
print(df[df['Price'] <= 0].head(5))

70
      Invoice StockCode                     Description  Quantity  \
4674   489825     22076              6 RIBBONS EMPIRE          12   
6781   489998     48185             DOOR MAT FAIRY CAKE         2   
16107  490727         M                          Manual         1   
18738  490961     22065  CHRISTMAS PUDDING TRINKET POT          1   
18739  490961     22142    CHRISTMAS CRAFT WHITE FAIRY         12   

               InvoiceDate  Price  Customer ID         Country  
4674   2009-12-02 13:34:00    0.0        16126  United Kingdom  
6781   2009-12-03 11:19:00    0.0        15658  United Kingdom  
16107  2009-12-07 16:38:00    0.0        17231  United Kingdom  
18738  2009-12-08 15:25:00    0.0        14108  United Kingdom  
18739  2009-12-08 15:25:00    0.0        14108  United Kingdom  


In [ ]:
df = df[df["Price"] > 0]

In [ ]:
# AFTER Removing zero/negative prices
print((df['Price'] <= 0).sum())
print(df[df['Price'] <= 0].head(10))

0
Empty DataFrame
Columns: [Invoice, StockCode, Description, Quantity, InvoiceDate, Price, Customer ID, Country]
Index: []


In [ ]:
# InvoiceDate is a datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [ ]:
import io

# your bucket name
bucket_name = "cust-seg-ygp"

# your output filename in S3
file_name = "cleaned_online_retail_II.csv"

# convert dataframe to in-memory CSV
csv_buffer = io.StringIO()
df.to_csv(csv_buffer, index=False)

# upload to S3
s3 = boto3.resource('s3')
s3.Object(bucket_name, file_name).put(Body=csv_buffer.getvalue())

print(f"Saved cleaned file to s3://{bucket_name}/{file_name}")

Saved cleaned file to s3://cust-seg-ygp/cleaned_online_retail_II.csv


In [ ]:
df.head(5)